# EKS Ops Agent Workshop

Build an AI agent that manages and troubleshoots Amazon EKS clusters using LangGraph, MCP Server and kagent.

| Module | Description | Dependency |
|--------|-------------|------------|
| **Module 0** | Enable kagent on existing EKS cluster | |
| **Module 1** | Barebone agent — Build and deploy BYO agent with Amazon Bedrock | Module 0 |
| **Module 2** | EKS MCP Server integration — Connect to 20+ cluster operations tools | Module 1 |
| **Module 3** | Memory — Persistent Redis memory for user defaults | Module 1 or 2 |

## Prerequisites

* This notebook **must be run** on the ML Ops desktop created via [quick start setup](../../../README.md#quick-start-basic). This notebook can also be used on an ML Ops desktop created via [advanced setup](../../../README.md#advanced-setup), but the user will need to appropriately adapt the notebook for use with the advanced setup.
* **AWS account with Bedrock access**: Enable model access in [AWS Bedrock Console](https://console.aws.amazon.com/bedrock/) → Model access. Default model: Claude Sonnet 4.


## Architecture

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                              EKS Cluster                                     │
│                                                                              │
│  ┌────────────────────────────────────────────────────────────────────────┐  │
│  │                         kagent namespace                               │  │
│  │                                                                        │  │
│  │   ┌─────────────┐     ┌───────────────────┐     ┌───────────────────┐  │  │
│  │   │  kagent-ui  │◄───►│ kagent-controller │◄───►│   eks-ops-agent   │  │  │
│  │   │    (Pod)    │     │       (Pod)       │     │       (Pod)       │  │  │
│  │   └─────────────┘     └─────────┬─────────┘     └─────────┬─────────┘  │  │
│  │                                 │                         │            │  │
│  │                       ┌─────────▼─────────┐               │            │  │
│  │                       │   PostgreSQL /    │               │            │  │
│  │                       │   SQLite (DB)     │               │            │  │
│  │                       └───────────────────┘               │            │  │
│  │                                                           │            │  │
│  │                       ┌───────────────────┐               │            │  │
│  │                       │  Redis (Module 3) │◄──────────────┘            │  │
│  │                       │  (User Defaults)  │                            │  │
│  │                       └───────────────────┘                            │  │
│  └────────────────────────────────────────────────────────────────────────┘  │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘
                                      │
          ┌───────────────────────────┼───────────────────────────┐
          │                           │                           │
          ▼                           ▼                           ▼
 ┌─────────────────┐        ┌─────────────────┐        ┌─────────────────┐
 │  Amazon Bedrock │        │   EKS MCP       │        │  Kubernetes     │
 │    (Claude)     │        │   Server        │        │     API         │
 └─────────────────┘        └─────────────────┘        └─────────────────┘
```

---
## Module 0: Enable kagent on Existing EKS Cluster

Your EKS cluster is already running (provisioned by the quick start template). This module enables kagent by re-applying Terraform with kagent variables. The quick start template created a `basic.tfvars` file with your cluster configuration (region, cluster name, AZs, S3 backend). We add a `workshop.tfvars` to enable kagent.

### Step 0.1: Set Up Paths

Define the directory paths used throughout the notebook.

In [ ]:
import os

REPO_DIR = os.path.expanduser("~/amazon-eks-machine-learning-with-terraform-and-kubeflow")
AGENT_DIR = os.path.join(REPO_DIR, "examples/agentic/eks-ops-agent")
TF_DIR = os.path.join(REPO_DIR, "eks-cluster/terraform/aws-eks-cluster-and-nodegroup")

os.environ["REPO_DIR"] = REPO_DIR
os.environ["AGENT_DIR"] = AGENT_DIR
os.environ["TF_DIR"] = TF_DIR

print(f"Repo Dir:  {REPO_DIR}")
print(f"Agent Dir: {AGENT_DIR}")
print(f"TF Dir:    {TF_DIR}")

### Step 0.2: Create Terraform Variables for kagent

Create a `workshop.tfvars` file that enables kagent on the existing cluster.

In [ ]:
%%bash
cat <<'EOF' > $TF_DIR/workshop.tfvars
# Enable kagent
kagent_enabled               = true
kagent_database_type         = "sqlite"
kagent_enable_ui             = true
kagent_enable_bedrock_access = true
EOF
echo "Created workshop.tfvars"

### Step 0.3: Apply Terraform

Apply Terraform with `basic.tfvars` (automatically created during quick start setup) and `workshop.tfvars` (kagent). This takes **5-10 minutes**. 

**Note:** If you used [advanced setup](../../../README.md#advanced-setup), you will need to apply terraform by using `workshop.tfvars` with your baseline `terraform apply` command.

In [ ]:
%%bash
cd $TF_DIR
terraform apply -auto-approve \
  -var-file=basic.tfvars \
  -var-file=workshop.tfvars

---
## Module 1: Build and Deploy Q&A Agent

Build and deploy a simple Q&A agent that answers Kubernetes and EKS questions using Amazon Bedrock Claude.

### Step 1.1: Build, Push, and Deploy

Build the Docker image, push to ECR, and deploy via Helm. The script reads the Bedrock IAM role ARN from Terraform output and configures IRSA automatically.

In [ ]:
%%bash
cd $AGENT_DIR
bash ./build-and-deploy.sh

### Step 1.2: Access the kagent UI

Forward the kagent UI port to localhost. Run the `kubectl port-forward` command in a **separate terminal** on the ML Ops desktop, then access the UI at `http://localhost:8080`.

In [ ]:
%%bash
echo "Run the following command in a separate terminal on the ML Ops desktop:"
echo ""
echo "  kubectl port-forward svc/kagent-ui -n kagent 8080:8080"
echo ""
echo "Then open: http://localhost:8080"
echo ""

### Step 1.3: Test the Agent

Open the kagent UI (port 8080), select **eks-ops-agent**, and try these prompts:

- "What is a Kubernetes Pod?"
- "How do I troubleshoot a CrashLoopBackOff error?"
- "Explain the difference between a Deployment and a StatefulSet"

The agent responds with Kubernetes/EKS guidance but cannot access your actual cluster yet — that comes in Module 2.

---
## Module 2: EKS MCP Server Integration

Enable 20+ tools from the [AWS managed EKS MCP Server](https://docs.aws.amazon.com/eks/latest/userguide/eks-mcp.html) that allow the agent to query and manage your actual EKS cluster.

The MCP integration code is already in `src/tools.py`. Setting `ENABLE_MCP_TOOLS=true` tells the agent to load these tools at startup.

### Step 2.1: Understand the Code

The MCP integration is in `src/tools.py`. Key points:

- Uses `langchain_mcp_adapters.client.MultiServerMCPClient` to connect to EKS MCP Server
- Spawns `mcp-proxy-for-aws` as a subprocess via stdio transport
- IRSA credentials are passed through automatically

```python
# src/tools.py
def get_mcp_server_config() -> dict:
    eks_mcp_endpoint = f"https://eks-mcp.{config.AWS_REGION}.api.aws/mcp"
    return {
        "eks-mcp": {
            "transport": "stdio",
            "command": "uvx",
            "args": ["mcp-proxy-for-aws@latest", eks_mcp_endpoint,
                     "--service", "eks-mcp", "--region", config.AWS_REGION],
        }
    }
```

### Step 2.2: Enable MCP Tools and Redeploy

Re-run the build and deploy script with `ENABLE_MCP_TOOLS=true`.

In [ ]:
%%bash
cd $AGENT_DIR
ENABLE_MCP_TOOLS=true bash ./build-and-deploy.sh

### Step 2.3: Verify Tools Loaded

You should see **"Creating agent with 20 tools"** in the logs.

In [ ]:
!kubectl logs -n kagent -l kagent=eks-ops-agent --tail=20

### Step 2.4: Test MCP Tools

Open the kagent UI and try these prompts (replace `<cluster-name>` with your actual cluster name):

1. **List resources:**
   ```
   List all pods in the kagent namespace on cluster <cluster-name>
   ```

2. **Get pod logs:**
   ```
   Get the logs from pod <pod-name> in namespace <namespace> on cluster <cluster-name>
   ```

3. **Generate and deploy manifests:**
   ```
   Generate a deployment manifest for a Redis application with 2 replicas.
   Deploy the manifest, ensure that Pods are in Running state.
   ```

4. **Get cluster insights:**
   ```
   Get insights and recommendations for cluster <cluster-name>
   ```

5. **Multi-step deployment task:**
   ```
   On cluster <cluster-name>, deploy a new nginx application called "test-nginx" with 3 replicas
   in the default namespace. After deployment, verify all pods are running.
   Then scale it down to 1 replica and confirm the change.
   ```

6. **Cluster discovery** (agent discovers clusters automatically):
   ```
   List all pods in the default namespace
   ```

### Sample Troubleshooting Scenario

**Step 1:** In the kagent UI, ask:
```
Deploy an nginx app called "broken-app" using image "nginx:doesnotexist" with 2 replicas
to the default namespace on cluster <cluster-name>
```

**Step 2:** Then ask:
```
The broken-app pods are not running. Investigate why and tell me how to fix it.
```

The agent will use multiple tools to diagnose: `list_k8s_resources`, `get_k8s_events`, `search_eks_troubleshooting_guide`, and provide actionable fix recommendations.

---
## Module 3: Memory with Redis

Enable memory that persists user defaults (cluster, namespace) across chat sessions.

The memory code is already in `src/memory.py`. Setting `ENABLE_MEMORY=true` deploys an in-cluster Redis instance and enables this feature.

```
Session 1:
  User: "Set my default cluster to <cluster-name> and namespace to default"
  Agent: Saved defaults

Session 2 (new session):
  User: "List all pods"
  Agent: (retrieves defaults from Redis, uses <cluster-name>/default)
```

> **Note:** This module uses an in-cluster Redis deployment for simplicity. For production, consider Amazon ElastiCache, MemoryDB, OpenSearch, or DynamoDB.

### Step 3.1: Enable Memory and Redeploy

Re-run the build and deploy script with both `ENABLE_MCP_TOOLS=true` and `ENABLE_MEMORY=true`. This automatically deploys Redis alongside the agent.

In [ ]:
%%bash
cd $AGENT_DIR
ENABLE_MCP_TOOLS=true ENABLE_MEMORY=true bash ./build-and-deploy.sh

### Step 3.2: Verify Memory Loaded

You should see **"Creating agent with 23 tools"** (20 MCP + 3 memory).

In [ ]:
!kubectl logs -n kagent -l kagent=eks-ops-agent --tail=20

### Step 3.3: Test Memory

Open the kagent UI and try these prompts (replace `<cluster-name>` with your actual cluster name):

1. **Set defaults:**
   ```
   Set my default cluster to <cluster-name> and namespace to default
   ```

2. **Verify defaults saved:**
   ```
   What are my defaults?
   ```

3. **Start a new chat session** (click "New Chat" in kagent UI)

4. **Use defaults implicitly:**
   ```
   List all pods
   ```
   The agent should use your saved cluster and namespace without asking.

5. **Clear defaults:**
   ```
   Clear my defaults
   ```

---
## Troubleshooting

| Error | Cause | Solution |
|-------|-------|----------|
| `AccessDenied on InvokeModel` | Bedrock model not enabled | Enable model access in [Bedrock console](https://console.aws.amazon.com/bedrock/) |
| `AccessDenied on InvokeModel` (Claude 4.x) | Missing inference profile prefix | Use `us.` prefix (e.g., `us.anthropic.claude-sonnet-4-...`) |
| `MCP tools not loading` | Environment variable not set | Verify `ENABLE_MCP_TOOLS=true` in Helm values |
| `MCP tools not loading` | Missing IAM permissions | Check IAM role has `eks-mcp:*` permissions |
| `Tool calls failing with Unauthorized` | Missing EKS Access Entry | Run `aws eks list-access-entries --cluster-name <cluster>` to verify |
| `IRSA not working` | ServiceAccount not annotated | Check: `kubectl get sa eks-ops-agent -n kagent -o yaml` |

In [ ]:
# Useful troubleshooting commands
!kubectl get agents -n kagent
!kubectl get pods -n kagent -l kagent=eks-ops-agent
!kubectl logs -n kagent -l kagent=eks-ops-agent --tail=50

---
## Cleanup

Uninstall the agent (includes Redis if deployed).

In [ ]:
%%bash
helm uninstall eks-ops-agent -n kagent
echo "Cleanup complete"